# Практическая работа № 1
## Рефакторинг текстовой игры в объектно-ориентированный код

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 1 — Введение. Повторение ООП-основ на Python  
**Продолжительность:** 4 ч аудиторно + 4 ч самостоятельно  
**Формат сдачи:** Git-репозиторий + UML + тесты + README + ADR  
**Максимальный балл:** 5 (4.5 + бонус за своевременность)

---

## О чём эта работа

Практическая работа № 1 — это **точка перехода** от процедурного мышления к объектному. Мы не пишем новую игру с нуля. Мы берём **уже работающий процедурный код** и переписываем его в ООП-стиле. Такой подход выбран не случайно:

1. **У вас есть эталон поведения.** Старый код работает — значит, мы точно знаем, чего хотим добиться от нового.
2. **Вы видите разницу.** Сравнивая «до» и «после», вы понимаете, что именно даёт ООП.
3. **Вы учитесь рефакторингу.** В реальной работе вы почти всегда будете работать с чужим кодом, а не писать с нуля.
4. **Вы осваиваете инструменты.** venv, pip, Git, pytest, PEP 8, UML, ADR — всё это стандарт индустрии, и вы будете использовать их в каждой следующей работе.

### Почему для игрового курса выбрана текстовая игра

Мы могли бы сразу взять Pygame и делать платформер. Но текстовая игра даёт **чистую архитектуру без визуального шума**: нет рендеринга, нет физики, нет спрайтов — только логика и состояние. На ней легче увидеть, как классы разделяют ответственности. Pygame мы подключим в модуле 4, когда будем изучать паттерн Game Loop.

### Чему вы научитесь

После выполнения этой работы вы сможете:
- анализировать процедурный код и выделять в нём сущности;
- проектировать классы с чёткой ответственностью;
- применять инкапсуляцию, свойства и инъекцию зависимостей;
- писать юнит-тесты с `pytest` и `Mock`;
- оформлять проект как настоящий репозиторий (README, ADR, .gitignore, requirements.txt);
- проверять стиль кода и вести осмысленную Git-историю.

---

## План работы

| Шаг | Что делаем | Что изучаем |
|---|---|---|
| 0 | Настройка окружения | venv, pip, Git, структура проекта |
| 1 | Анализ процедурного кода | декомпозиция предметной области |
| 2 | Проектирование классов и UML | отношения классов, диаграммы |
| 3 | Реализация классов | инкапсуляция, свойства, DI |
| 4 | Юнит-тесты | pytest, Mock, TDD-мышление |
| 5 | Документация | README, ADR |
| 6 | PEP 8 и финальный коммит | стиль, Git-гигиена |

In [ ]:
# Проверка версии Python (нужна 3.10+)
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

# Проверка установки зависимостей
for pkg in ["pytest", "pycodestyle"]:
    try:
        module = __import__(pkg)
        version = getattr(module, "__version__", "—")
        print(f"OK: {pkg} {version}")
    except ImportError:
        print(f"MISSING: {pkg} — установите: pip install {pkg}")

---
# Шаг 0. Настройка окружения

## Теория: зачем нужно окружение проекта

### 0.1. Виртуальное окружение (venv)

**Проблема:** разные проекты требуют разных версий библиотек. Проект A использует Django 3, проект B — Django 4. Если устанавливать всё глобально, проекты начнут конфликтовать.

**Решение:** виртуальное окружение — это **изолированная папка**, содержащая собственный интерпретатор Python и собственный набор пакетов. Внутри `.venv` живёт своя «мини-вселенная», не связанная с системным Python.

Когда вы активируете виртуальное окружение, команды `python` и `pip` указывают не на глобальные версии, а на локальные. Всё, что вы устанавливаете, попадает в `.venv`, а не в систему.

**Зачем это в курсе:** в модулях 2–8 вы будете ставить разные пакеты (`pygame`, `pytest`, `pytest-cov`, `mypy`, `flake8`). Без venv они смешаются в глобальной системе, и однажды что-то перестанет работать.

**Для углублённого изучения:**
- [Python Packaging Authority: venv](https://packaging.python.org/en/latest/guides/installing-using-pip-and-virtual-environments/)
- [Real Python: Python Virtual Environments](https://realpython.com/python-virtual-environments-a-primer/)

### 0.2. Менеджер пакетов pip и requirements.txt

**pip** — стандартный менеджер пакетов Python. Скачивает библиотеки с [PyPI](https://pypi.org/) и устанавливает их в активное окружение.

**`requirements.txt`** — текстовый файл со списком зависимостей проекта с точными версиями. Он нужен для воспроизводимости: любой разработчик (или вы сами через месяц) может выполнить `pip install -r requirements.txt` и получить **точно то же окружение**.

**Правило:** `requirements.txt` хранится в Git. Папка `.venv` — не хранится (она воссоздаётся из requirements).

**Для углублённого изучения:**
- [pip documentation](https://pip.pypa.io/en/stable/)
- [Real Python: Pip and requirements.txt](https://realpython.com/what-is-pip/)

### 0.3. Git как инструмент профессиональной работы

**Git** — распределённая система контроля версий. Она записывает изменения в файлах и позволяет:
- возвращаться к предыдущим версиям;
- работать в изолированных ветках (например, для экспериментов);
- координировать работу команды;
- видеть, кто, что и когда изменил.

**Зачем это в курсе:** все практические работы сдаются через Git-репозиторий. По истории коммитов преподаватель видит **процесс** — как вы шли к результату. Это важнее, чем финальный файл.

**Соглашение о сообщениях коммитов:** используйте повелительное наклонение. «Add player class» лучше, чем «Added player class» или «added». Подробнее: [Conventional Commits](https://www.conventionalcommits.org/).

**Для углублённого изучения:**
- [Software Carpentry: Version Control with Git](https://swcarpentry.github.io/git-novice/)
- [Pro Git Book (на русском)](https://git-scm.com/book/ru/v2)

## Практика: команды для терминала

Выполните в терминале (не в блокноте) один раз перед началом работы:

```bash
# 1. Создать папку проекта
mkdir guess_number && cd guess_number

# 2. Создать виртуальное окружение
python -m venv .venv

# 3. Активировать (выбрать одну строку для вашей ОС)
# Windows PowerShell:    .venv\Scripts\Activate.ps1
# Windows cmd:           .venv\Scripts\activate.bat
# macOS / Linux:         source .venv/bin/activate

# 4. Установить зависимости
pip install pytest pycodestyle

# 5. Зафиксировать версии
pip freeze > requirements.txt

# 6. Инициализировать Git
git init
git config user.name "Ваше Имя"
git config user.email "your.email@example.com"
```

Создайте файл `.gitignore` с содержимым:

```
.venv/
__pycache__/
*.pyc
.pytest_cache/
*.egg-info/
```

Это исключит из репозитория временные файлы. **Отметка для особого внимания:** без `.gitignore` в репозиторий попадут сотни файлов из `.venv`, и он раздуется до десятков мегабайт.

---
# Шаг 1. Анализ процедурного кода

## Теория: как перейти от процедурного мышления к объектному

### 1.1. Что такое процедурный код

**Процедурный код** организован вокруг **действий**: есть функции, которые выполняют операции над данными. Данные передаются в функции в виде аргументов или хранятся в глобальных переменных.

Типичная процедурная игра «Угадай число» выглядит так:
```python
secret = random.randint(1, 100)
attempts = 0
while attempts < 10:
    guess = int(input())
    if guess < secret: ...
```

Здесь нет классов — только переменные и функции. Это работает, но плохо масштабируется.

### 1.2. Что такое объектный код

**Объектный код** организован вокруг **сущностей**: есть классы, у каждого — своё состояние и поведение. Взаимодействие происходит через вызов методов.

Та же игра в ООП-стиле:
```python
game = Game()
game.run()
```

Внутри `Game.run()` создаётся объект `Player`, взаимодействует с `UI`, управляет попытками. Каждый класс отвечает за своё.

### 1.3. Как проводить декомпозицию

**Декомпозиция** — это разбиение задачи на части. В ООП декомпозиция проводится по **сущностям**, а не по действиям.

**Алгоритм анализа процедурного кода:**

1. **Прочитайте код сверху вниз.** Постарайтесь понять, что происходит.
2. **Выпишите все данные.** Переменные, аргументы, глобальные значения.
3. **Сгруппируйте данные по смыслу.** Какие данные относятся к игроку? К игровому полю? К интерфейсу?
4. **Выпишите все действия.** Функции, циклы, условия.
5. **Сгруппируйте действия по данным.** Какие действия работают с состоянием игры? Какие — с вводом/выводом?
6. **Каждая группа = потенциальный класс.**

**Ключевой принцип:** класс должен иметь **одну ответственность**. Если вы не можете описать класс одним предложением без союза «и» — вероятно, стоит разделить его.

**Пример декомпозиции для «Угадай число»:**

| Данные | Действия | Класс |
|---|---|---|
| `secret`, `attempts`, `max_attempts` | генерация числа, проверка попытки, игровой цикл | `Game` |
| `name`, `attempts` | запросить ввод, увеличить счётчик | `Player` |
| — | `print`, `input` | `UI` |

### 1.4. Признаки плохой декомпозиции

- **God Object** — один класс делает всё.
- **Anemic Domain Model** — классы содержат только данные, вся логика снаружи.
- **Feature Envy** — метод одного класса активно использует данные другого.
- **Data Clumps** — одни и те же переменные всегда передаются вместе (например, `x, y, z`).

**Для углублённого изучения:**
- Martin Fowler. «Refactoring» — главы о декомпозиции и code smells.
- [Refactoring Guru: Code Smells](https://refactoring.guru/refactoring/smells)

## Практика: анализ исходного кода

Ниже — процедурная реализация игры «Угадай число». Прочитайте её и **проведите декомпозицию** по алгоритму выше.

In [ ]:
# Исходный процедурный код (только для чтения и анализа — не запускаем input)
import random

def play_guess_number_procedural():
    secret = random.randint(1, 100)
    attempts = 0
    max_attempts = 10
    print("Я загадал число от 1 до 100. У вас 10 попыток.")
    while attempts < max_attempts:
        try:
            guess = int(input("Ваш вариант: "))
        except ValueError:
            print("Введите число!")
            continue
        attempts += 1
        if guess < secret:
            print("Больше!")
        elif guess > secret:
            print("Меньше!")
        else:
            print(f"Угадали! Попыток: {attempts}")
            return
    print(f"Попытки закончились. Было загадано: {secret}")

print("Функция определена, готова к анализу")

### Задание 1.1. Таблица анализа

Заполните таблицу для вашей игры (не обязательно «Угадай число» — можно взять «Крестики-нолики», «Виселицу», «Камень-ножницы-бумага»):

| Что | Где в исходном коде | Куда перенести |
|---|---|---|
| ... | ... | ... |

### Задание 1.2. Ответьте письменно

1. Какие классы вы выделите? Обоснуйте.
2. Какие атрибуты будут приватными (`_name` или `__name`)? Почему?
3. Какие методы будут публичными? Почему?
4. Есть ли в исходном коде «God Object»? Как вы это поняли?

**Ответы сохраните в файл** `docs/analysis.md` — они станут частью документации проекта.

---
# Шаг 2. Проектирование классов и UML

## Теория: зачем проектировать до кодирования

### 2.1. Что такое UML и зачем он нужен

**UML** (Unified Modeling Language) — это **стандартный язык визуального моделирования** программных систем. Он позволяет описать структуру классов, их атрибуты, методы и отношения, не написав ни строчки кода.

**Зачем UML в курсе:**
- **Наглядность.** Диаграмма классов показывает архитектуру за 10 секунд, а чтение кода займёт 10 минут.
- **Планирование.** Ошибка на диаграмме стоит 5 минут, а ошибка в коде — часы рефакторинга.
- **Коммуникация.** Диаграмму можно показать коллеге, тимлиду, заказчику.
- **Документация.** Диаграмма в README позволяет новому разработчику быстро понять проект.

### 2.2. Диаграмма классов: обозначения

**Класс** изображается прямоугольником с тремя секциями:
```
+------------------+
|      Game        |  ← имя класса
+------------------+
| - secret: int    |  ← атрибуты
| - max_attempts: int |
+------------------+
| + run(): void    |  ← методы
| + check_guess(): str |
+------------------+
```

**Модификаторы доступа:**
- `+` — публичный (public).
- `-` — приватный (private).
- `#` — защищённый (protected).

**Отношения между классами:**

| Отношение | Обозначение | Пример | Смысл |
|---|---|---|---|
| Наследование | полый треугольник | `Player → GameObject` | «является» |
| Композиция | закрашенный ромб | `Game ◆— Player` | «содержит (жизненный цикл)» |
| Агрегация | полый ромб | `Game ◇— Player` | «содержит (слабая связь)» |
| Ассоциация | линия | `Player — UI` | «использует» |
| Зависимость | пунктирная стрелка | `Game ⤳ Config` | «зависит от» |

**Правило:** если объект A не существует без B — это композиция (ромб). Если A использует B, но может работать и без него — ассоциация (линия).

### 2.3. Инструменты для диаграмм

- **draw.io** — бесплатный онлайн-редактор, экспорт в PNG/SVG.
- **PlantUML** — текстовое описание, версионируется в Git.
- **Mermaid** — текстовое описание, встроено в GitHub и Markdown.

**Рекомендация курса:** используйте **Mermaid**. Причины:
- диаграмма хранится как текст → её можно версионировать;
- GitHub и VS Code рендерят Mermaid автоматически;
- не нужно устанавливать дополнительный софт.

**Для углублённого изучения:**
- [UML Diagrams](https://www.uml-diagrams.org/class-diagrams-overview.html)
- [Mermaid documentation](https://mermaid.js.org/syntax/classDiagram.html)

## Практика: создание диаграммы

Ниже — Mermaid-диаграмма для эталонного решения. Скопируйте её в `README.md`, если решите следовать этой архитектуре. Или адаптируйте под свою игру.

In [ ]:
uml = '''
```mermaid
classDiagram
    class Game {
        -secret: int
        -max_attempts: int
        -ui: UI
        +run(): void
        +check_guess(guess: int): str
        +secret: int
    }
    class Player {
        -name: str
        -attempts: int
        -ui: UI
        +make_guess(): int
        +increment_attempts(): void
        +name: str
        +attempts: int
    }
    class UI {
        +show_message(message: str): void
        +get_input(prompt: str): str
    }
    Game --> UI : uses
    Game --> Player : creates
    Player --> UI : uses
```
'''
print(uml)

# Сохраним для дальнейшего использования
import os
os.makedirs('docs', exist_ok=True)
with open('docs/uml.md', 'w', encoding='utf-8') as f:
    f.write(uml)
print("✅ Сохранено в docs/uml.md")

### Что важно на диаграмме

**Обратите внимание:**

1. **`Game` создаёт `Player`** (стрелка `creates`). Это означает, что жизненный цикл `Player` управляется `Game`. В терминах UML это ближе к композиции.
2. **`Game` использует `UI`** (стрелка `uses`). `UI` создаётся извне и передаётся в конструктор — это **инъекция зависимостей** (см. шаг 3).
3. **`Player` использует `UI`** — тоже через конструктор.
4. **Все атрибуты приватные** (`-`), доступ — через свойства (`+ secret`, `+ name`, `+ attempts`).

### Задание 2.1

Скорректируйте диаграмму под свою игру. Ответьте:
1. Есть ли у вас наследование? Если да — обоснуйте.
2. Есть ли композиция? Что чем владеет?
3. Есть ли циклические зависимости? (A использует B, B использует A) — если да, это повод задуматься.

Сохраните ответы в `docs/adr_002_diagram.md`.

---
# Шаг 3. Реализация классов

## Теория: инкапсуляция, свойства и инъекция зависимостей

### 3.1. Инкапсуляция на практике

**Инкапсуляция** — это сокрытие внутреннего состояния объекта и предоставление контролируемого доступа через публичные методы.

**В Python инкапсуляция реализована через соглашения:**
- `_name` — защищённый. «Не трогайте снаружи, если не понимаете, что делаете».
- `__name` — приватный (name mangling). Python переименует в `_ClassName__name`.
- `name` — публичный.

**Почему не `private`/`public` как в Java:** Python — «язык для взрослых». Считается, что программист сам способен соблюдать границы. Если он залез в `_name` — это его ответственность.

**Что инкапсулировать в игре:**
- Текущее здоровье, ману, очки.
- Координаты, скорость.
- Счётчики попыток, очки.
- Внутренние буферы, кэши.

**Что делать публичным:**
- Константы, не изменяемые после инициализации.
- Публичные методы интерфейса (`take_damage`, `move`).
- Свойства только для чтения (`@property` без сеттера).

### 3.2. Свойства (`@property`)

**Проблема:** если сделать атрибут публичным, внешний код может присвоить некорректное значение:
```python
player.health = -1000  # баг
```

**Наивное решение (Java-стиль):**
```python
def get_health(self): ...
def set_health(self, value): ...
```

Минус: громоздко. `player.get_health()` вместо `player.health`.

**Python-решение — свойства:**
```python
@property
def health(self):
    return self._health

@health.setter
def health(self, value):
    self._health = max(0, min(value, self._max_health))
```

Снаружи используется как атрибут (`player.health = 50`), но внутри работает валидация.

**Когда использовать `@property`:**
- когда нужна валидация;
- когда значение вычисляется на лету;
- когда нужна совместимость со старым API;
- когда нужна ленивая инициализация.

**Когда НЕ использовать:**
- если атрибут просто хранит значение и не требует валидации — оставьте публичным;
- если логика сложная — вынесите в отдельный метод.

### 3.3. Инъекция зависимостей (Dependency Injection)

**Проблема:** класс `Player` должен получать ввод от пользователя. Если он вызовет `input()` напрямую, его нельзя будет протестировать — тест «зависнет» на ожидании ввода.

**Решение:** вынести ввод в отдельный класс `UI` и **передавать его в `Player` через конструктор**:
```python
class Player:
    def __init__(self, name, ui):
        self._ui = ui
```

Теперь в тестах можно передать **Mock** вместо реального UI:
```python
ui = Mock()
ui.get_input.return_value = "42"
player = Player("Test", ui)
```

**Три способа инъекции:**
1. **Через конструктор** (используем в курсе) — самый явный.
2. **Через setter** — для изменяемых во времени зависимостей.
3. **Через параметр метода** — для одноразовых зависимостей.

**Для углублённого изучения:**
- [Martin Fowler: Inversion of Control](https://martinfowler.com/articles/injection.html)
- [Real Python: Python's property()](https://realpython.com/python-property/)
- [Real Python: Dependency Injection](https://realpython.com/dependency-injection-python/)

## Практика: реализация классов

Создайте структуру проекта:
```
guess_number/
├── src/
│   ├── __init__.py
│   ├── game.py
│   ├── player.py
│   └── ui.py
├── tests/
├── docs/
├── main.py
├── requirements.txt
├── .gitignore
└── README.md
```

Ниже — эталонная реализация. В реальном проекте каждый класс лежит в отдельном модуле. Здесь мы используем `%%writefile` (работает в Jupyter, JupyterLab, VS Code, PyCharm), чтобы ячейка создавала файл на диске.

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('tests', exist_ok=True)
os.makedirs('docs', exist_ok=True)
print("✅ Папки созданы")

In [ ]:
%%writefile src/__init__.py
"""Пакет с игровой логикой."""


In [ ]:
%%writefile src/ui.py
"""Интерфейс пользователя."""


class UI:
    """Абстракция над вводом/выводом.

    Инкапсулирует ввод и вывод, чтобы:
    1. Не засорять игровую логику вызовами print/input.
    2. Позволить подменять UI в тестах (Mock) и демонстрациях (FakeUI).
    """

    def show_message(self, message: str) -> None:
        """Показать сообщение пользователю."""
        print(message)

    def get_input(self, prompt: str) -> str:
        """Получить ввод от пользователя."""
        return input(prompt)


In [ ]:
%%writefile src/player.py
"""Игрок."""

from .ui import UI


class Player:
    """Игрок: имя, счётчик попыток, взаимодействие с UI.

    Атрибуты:
        _name (str): имя игрока (инкапсулировано).
        _attempts (int): количество сделанных попыток.
        _ui (UI): зависимость, инжектируется через конструктор.
    """

    def __init__(self, name: str, ui: UI):
        self._name = name
        self._attempts = 0
        self._ui = ui

    @property
    def name(self) -> str:
        return self._name

    @property
    def attempts(self) -> int:
        return self._attempts

    def make_guess(self) -> int:
        """Запросить у игрока целое число.

        Повторяет запрос при неверном вводе.
        """
        while True:
            try:
                return int(self._ui.get_input("Ваш вариант: "))
            except ValueError:
                self._ui.show_message("Введите целое число!")

    def increment_attempts(self) -> None:
        """Увеличить счётчик попыток на 1."""
        self._attempts += 1


In [ ]:
%%writefile src/game.py
"""Игра «Угадай число»."""

import random
from .ui import UI
from .player import Player


class Game:
    """Управляет игровым циклом «Угадай число».

    Отвечает за:
    - генерацию секретного числа;
    - создание игрока;
    - проверку попыток;
    - завершение игры.
    """

    def __init__(self, max_attempts: int = 10, ui: UI | None = None):
        self._max_attempts = max_attempts
        self._secret = 0
        self._ui = ui if ui is not None else UI()

    def run(self) -> None:
        """Запустить игровой цикл."""
        self._secret = random.randint(1, 100)
        self._ui.show_message(
            f"Я загадал число от 1 до 100. У вас {self._max_attempts} попыток."
        )
        player = Player("Игрок", self._ui)
        while player.attempts < self._max_attempts:
            guess = player.make_guess()
            player.increment_attempts()
            result = self.check_guess(guess)
            self._ui.show_message(result)
            if guess == self._secret:
                return
        self._ui.show_message(
            f"Попытки закончились. Было загадано: {self._secret}"
        )

    def check_guess(self, guess: int) -> str:
        """Вернуть сообщение-подсказку для попытки."""
        if guess < self._secret:
            return "Больше!"
        elif guess > self._secret:
            return "Меньше!"
        return "Угадали!"

    @property
    def secret(self) -> int:
        """Секретное число (только чтение; используется в тестах)."""
        return self._secret


In [ ]:
%%writefile main.py
"""Точка входа в игру."""

from src.game import Game


def main() -> None:
    Game(max_attempts=10).run()


if __name__ == "__main__":
    main()


### Что здесь важно

1. **`Player.__init__` принимает `ui`** — это инъекция зависимости. В тестах подставим Mock.
2. **`Game.__init__` принимает `ui`** — тоже DI. Если `ui` не передан, создаётся настоящий `UI()` (значение по умолчанию через `None` — безопасный паттерн).
3. **`_secret`, `_max_attempts`, `_ui` — приватные.** Внешний код работает только через `run()`, `check_guess()`, `secret`.
4. **`secret` — свойство только для чтения.** Тесты могут узнать значение, но не изменить.
5. **`make_guess()` повторяет ввод** при ошибке — это устойчивость к пользовательским ошибкам.

### Демонстрация: игра с автоматическим вводом

In [ ]:
# Подставим FakeUI с заранее заданными ответами, чтобы не ждать ввода
from src.ui import UI
from src.game import Game


class FakeUI(UI):
    """UI для демонстрации: заранее заданные ответы."""

    def __init__(self, answers):
        self._answers = iter(answers)
        self.messages = []

    def get_input(self, prompt: str) -> str:
        return next(self._answers)

    def show_message(self, message: str) -> None:
        self.messages.append(message)
        print(message)


fake_ui = FakeUI(["50", "25", "37", "42"])
game = Game(max_attempts=5, ui=fake_ui)
game._secret = 42  # фиксируем секрет для воспроизводимости
game.run()
print("\nСохранённые сообщения UI:", fake_ui.messages)

---
# Шаг 4. Юнит-тесты с pytest

## Теория: зачем писать тесты

### 4.1. Что такое юнит-тест

**Юнит-тест** — это автоматическая проверка отдельной единицы кода (обычно функции или метода). Юнит-тесты:
- **быстрые** — запускаются за миллисекунды;
- **изолированные** — не зависят от других тестов;
- **детерминированные** — всегда дают одинаковый результат;
- **автоматические** — не требуют ручного вмешательства.

### 4.2. Зачем писать тесты в геймдеве

Игры — это сложные системы с множеством состояний. Тесты позволяют:
- **Ловить регрессии** — если после рефакторинга что-то сломалось, тест сразу сообщит.
- **Документировать поведение** — тест показывает, как использовать API.
- **Ускорять разработку** — не нужно вручную проверять каждую функцию после каждого изменения.
- **Улучшать дизайн** — если функцию сложно тестировать, вероятно, у неё плохой интерфейс.

**Отметка для особого внимания:** в играх тестируют не всё. Отрисовку, физику и баланс обычно проверяют вручную. Но **игровая логика** (правила, состояния, экономика) — идеальный кандидат для тестов.

### 4.3. Что такое Mock

**Mock** — это объект-заглушка, имитирующий поведение реального объекта. Он нужен, чтобы **изолировать** тестируемый код от внешних зависимостей (ввод-вывод, сеть, файлы).

**Пример:** мы хотим протестировать `Player.make_guess()`. Она вызывает `self._ui.get_input()`. Без Mock нам пришлось бы вводить данные вручную. С Mock мы говорим: «когда кто-то вызовет `get_input()`, верни `42`».

**Три способа настройки Mock:**
- `mock.return_value = X` — всегда возвращать X.
- `mock.side_effect = [X, Y, Z]` — возвращать значения по очереди.
- `mock.side_effect = Exception("...")` — выбрасывать исключение.

**Для углублённого изучения:**
- [pytest documentation](https://docs.pytest.org/)
- [Python: unittest.mock](https://docs.python.org/3/library/unittest.mock.html)
- [Real Python: Testing Your Code with pytest](https://realpython.com/pytest-python-testing/)

## Практика: написание тестов

Создайте файлы тестов:

In [ ]:
%%writefile tests/__init__.py
"""Тесты проекта."""


In [ ]:
%%writefile tests/test_game.py
"""Тесты класса Game."""

from src.game import Game


class TestGame:
    """Проверяем логику проверки попытки."""

    def test_check_guess_too_low(self):
        """Если попытка меньше секрета — «Больше!»."""
        g = Game()
        g._secret = 50
        assert g.check_guess(30) == "Больше!"

    def test_check_guess_too_high(self):
        """Если попытка больше секрета — «Меньше!»."""
        g = Game()
        g._secret = 50
        assert g.check_guess(70) == "Меньше!"

    def test_check_guess_correct(self):
        """Если попытка равна секрету — «Угадали!»."""
        g = Game()
        g._secret = 50
        assert g.check_guess(50) == "Угадали!"

    def test_secret_property(self):
        """Свойство secret возвращает целое число."""
        g = Game()
        assert isinstance(g.secret, int)


In [ ]:
%%writefile tests/test_player.py
"""Тесты класса Player."""

from unittest.mock import Mock
from src.player import Player


class TestPlayer:
    """Проверяем счётчик попыток и обработку ввода."""

    def test_initial_attempts(self):
        p = Player("Test", Mock())
        assert p.attempts == 0

    def test_increment_attempts(self):
        p = Player("Test", Mock())
        p.increment_attempts()
        assert p.attempts == 1

    def test_make_guess_valid(self):
        """Корректный ввод возвращается как int."""
        ui = Mock()
        ui.get_input.return_value = "42"
        p = Player("Test", ui)
        assert p.make_guess() == 42

    def test_make_guess_invalid_then_valid(self):
        """При неверном вводе запрос повторяется."""
        ui = Mock()
        ui.get_input.side_effect = ["abc", "42"]
        p = Player("Test", ui)
        assert p.make_guess() == 42
        assert ui.get_input.call_count == 2


### Запуск тестов

**Через терминал:**
```bash
pytest -v
```

**Прямо из блокнота:**

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-v', '--tb=short'],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)
print(f"Код выхода: {result.returncode}")

**Ожидаемый результат:** 8 тестов passed. Если что-то упало — читайте `--tb=short` (краткий traceback) и исправляйте.

---
# Шаг 5. Документация: README и ADR

## Теория: зачем нужна документация

### 5.1. README

**README.md** — это первое, что видит человек, открывший ваш репозиторий. Хороший README отвечает на вопросы:
- Что это за проект?
- Как его установить?
- Как его запустить?
- Как запустить тесты?
- Какая у него архитектура?

**Правило:** если человек не может запустить ваш проект, следуя README, — README плохой.

### 5.2. ADR (Architecture Decision Record)

**ADR** — короткий документ, фиксирующий **архитектурное решение**: контекст, рассмотренные альтернативы, принятое решение и последствия.

**Зачем ADR:** через месяц вы забудете, почему сделали так, а не иначе. Через год новый разработчик спросит: «Почему `UI` передаётся через конструктор, а не создаётся внутри?» — и ADR ответит.

**Структура ADR:**
1. **Название и номер** (ADR-001).
2. **Контекст** — какая проблема решается.
3. **Решение** — что выбрано.
4. **Рассмотренные альтернативы** — что ещё думали.
5. **Последствия** — плюсы и минусы.

**Формат хранения:** папка `docs/` или `docs/adr/` с файлами `001-xxx.md`, `002-yyy.md`.

**Для углублённого изучения:**
- [Michael Nygard: Documenting Architecture Decisions](https://cognitect.com/blog/2011/11/15/documenting-architecture-decisions)
- [GitHub: ADR templates](https://github.com/joelparkerhenderson/architecture-decision-record)

In [ ]:
%%writefile README.md
# Guess Number — ООП-версия

## Описание
Текстовая игра «Угадай число», переписанная с процедурного кода на ООП.

## Установка
```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# macOS/Linux: source .venv/bin/activate
pip install -r requirements.txt
```

## Запуск
```bash
python main.py
```

## Тесты
```bash
pytest -v
```

## Архитектура
- `Game` — управление игровым циклом.
- `Player` — состояние игрока и ввод.
- `UI` — абстракция ввода/вывода (подменяется в тестах).

## Диаграмма классов
См. `docs/uml.md`.

## Решения
- [ADR-001: Разделение Game и UI](docs/adr_001_ui.md)

## Структура
```
guess_number/
├── src/       # исходный код
├── tests/     # тесты
├── docs/      # документация
└── main.py    # точка входа
```


In [ ]:
%%writefile docs/adr_001_ui.md
# ADR-001: Разделение Game и UI

## Контекст
Процедурный код смешивал игровую логику и ввод/вывод. Это затрудняло тестирование:
тесты требовали ручного ввода данных.

## Решение
Выделен класс `UI`, отвечающий за ввод/вывод. `Game` и `Player`
получают `UI` через конструктор (инъекция зависимостей).

## Рассмотренные альтернативы
1. **Глобальный UI-модуль.** Проще, но сложнее тестировать и хуже изолирует.
2. **Наследование от UI.** Приводит к жёсткой связи и комбинаторному взрыву.
3. **Инъекция через конструктор (выбрано).** Гибко, тестируемо, соответствует DIP.

## Последствия
- ✅ Логика тестируется через Mock без реального ввода.
- ✅ Легко заменить консольный UI на GUI.
- ⚠️ +1 уровень косвенности — цена гибкости.
- ⚠️ Требуется явно передавать UI в конструкторы.


In [ ]:
# requirements.txt
with open('requirements.txt', 'w', encoding='utf-8') as f:
    f.write('pytest\npycodestyle\n')
print("✅ requirements.txt создан")

---
# Шаг 6. PEP 8 и финальный коммит

## Теория: PEP 8 и гигиена Git

### 6.1. Что такое PEP 8

**PEP 8** (Python Enhancement Proposal #8) — официальное руководство по стилю кода Python. Оно описывает:
- **отступы** — 4 пробела, не табы;
- **длину строк** — до 79 символов (в проектах допускается 100);
- **именование** — `snake_case` для функций и переменных, `CamelCase` для классов, `UPPER_CASE` для констант;
- **пробелы** — вокруг операторов, после запятых;
- **пустые строки** — 2 между функциями и классами, 1 внутри класса;
- **импорты** — по одному на строку, в начале файла.

**Почему это важно:** код читается в 10 раз чаще, чем пишется. Единый стиль позволяет всей команде понимать код без «переключения» между разными манерами.

**Автоматизация:** инструменты `pycodestyle`, `flake8`, `black`, `ruff` проверяют стиль автоматически. В курсе мы используем `pycodestyle` — самый простой.

**Для углублённого изучения:**
- [PEP 8 — Style Guide for Python Code](https://peps.python.org/pep-0008/)
- [Real Python: How to Write Beautiful Python Code](https://realpython.com/python-pep8/)

### 6.2. Гигиена Git

**Хорошая Git-история:**
- **Атомарные коммиты** — один коммит = одно логическое изменение.
- **Осмысленные сообщения** — «Add Player class», а не «fix».
- **Повелительное наклонение** — так принято в индустрии.
- **Не коммитить секреты** — пароли, токены, ключи.
- **Не коммитить артефакты** — `.venv`, `__pycache__`, IDE-файлы.

**Типичные ошибки новичков:**
- Один гигантский коммит «всё сразу».
- Сообщения вида «123», «fix», «update».
- Коммит `.venv/` или `venv/`.
- Коммит файлов IDE (`.idea/`, `.vscode/`).

**Для углублённого изучения:**
- [Conventional Commits](https://www.conventionalcommits.org/)
- [How to Write a Git Commit Message](https://cbea.ms/git-commit/)

In [ ]:
# Проверка стиля
result = subprocess.run(
    [sys.executable, '-m', 'pycodestyle', 'src/', 'tests/', 'main.py'],
    capture_output=True,
    text=True,
)
if result.stdout:
    print("⚠️ Найдены замечания PEP 8:")
    print(result.stdout)
else:
    print("✅ PEP 8: замечаний нет")

### Финальный коммит

В терминале:
```bash
git add .
git commit -m "Complete OOP refactoring with tests and docs"
git log --oneline
```

**Хорошая история коммитов для этой работы:**
```
Initial commit: project setup
Add UI class with input/output abstraction
Add Player class with encapsulated state
Add Game class with dependency injection
Add unit tests with pytest and Mock
Add README, ADR and UML diagram
Fix PEP 8 issues
```

Если вы коммитите всё одним сообщением — преподаватель не увидит процесса. Это снижает оценку.

---
# Самопроверка: чек-лист

In [ ]:
checklist = [
    "ООП-рефакторинг выполнен",
    "Классы разделены по ответственностям",
    "Инкапсуляция (_ и __) применена",
    "@property использован хотя бы один раз",
    "Есть инъекция зависимостей (UI)",
    "Игра запускается",
    "Написаны тесты pytest",
    "Используется Mock для UI",
    "README.md создан",
    "ADR создан",
    "UML-диаграмма создана",
    "PEP 8 проверен",
    "Git-история осмысленна",
]
print("Чек-лист для самопроверки:")
for i, item in enumerate(checklist, 1):
    print(f"[ ] {i:2d}. {item}")

---
# Критерии оценки

| Критерий | Баллы | Что проверяется |
|---|---:|---|
| Корректность ООП-рефакторинга | 1.5 | Классы выделены по ответственностям, инкапсуляция применена, нет «божественного объекта» |
| Работоспособность | 1.0 | Игра запускается, функциональность сохранена |
| Тесты (pytest + Mock) | 1.0 | Тесты проходят, используются Mock, покрытие ключевой логики |
| Документация (README, ADR, UML) | 0.5 | README содержит инструкции, ADR объясняет решения, UML корректен |
| Качество кода (PEP 8, Git) | 0.5 | Нет замечаний pycodestyle, осмысленная история коммитов |
| **Итого** | **4.5** | |

Максимум — 5 баллов при сдаче в срок. При задержке оценка снижается на 1 балл за каждую неделю.

---

# Домашнее задание к следующему модулю

1. Убедиться, что проект полностью работает локально.
2. Закоммитить все изменения в Git.
3. Прочитать:
   - Лутц М. «Изучаем Python», главы 26–27.
   - [Real Python: Abstract Base Classes](https://realpython.com/python-interface/)
   - [Python docs: dataclasses](https://docs.python.org/3/library/dataclasses.html)
4. Подготовить вопросы по темам:
   - `abc.ABC` и `@abstractmethod`.
   - `@dataclass` и её параметры (`frozen`, `field`).
   - `typing.Protocol`.
   - Миксины.
   - `__slots__`.

## Что дальше

В модуле 2 мы изучим современные возможности ООП на Python:
- абстрактные базовые классы для явных интерфейсов;
- `dataclasses` для компонентов и конфигураций;
- протоколы для структурной типизации;
- миксины для переиспользования поведения;
- `__slots__` для оптимизации памяти (важно для игр).

Всё это напрямую пригодится в вашем итоговом проекте.